# Regressão Logística com Validação de Premissas - Statsmodel

## Bibliotecas e Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Dependências
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados
import pandas as pd
import numpy as np
from datetime import datetime
import pickle
import os

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Modelos
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Métricas e validação
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, 
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print('✅ Ambiente Configurado')

Diretórios carregadas com sucesso
Funções básicas carregadas com sucesso
Funções extras carregadas com sucesso
✅ Ambiente Configurado


## Parâmetros Globais

In [3]:
# Definições
TARGET = 'FPD'
RANDOM_STATE = 42
IV = 0.001
THRESHOLD = 0.5
C = 0.1
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
VERSAO = 'V1-4 - RL(Statsmodel COM Premissas)'

## Carregamento dos Dados

In [4]:
# Carregar dados CORRIGIDOS conforme solicitado
train = pd.read_csv(PROCESSED_DIR / 'abt01_train.csv')
test = pd.read_csv(PROCESSED_DIR / 'abt01_test.csv')

print(f'📊 Treino: {train.shape}')
print(f'📊 Teste: {test.shape}')
print(f'\n🎯 Distribuição do Target (Treino):')
print(f"  Treino: {(train[TARGET].value_counts(normalize=True) * 100).round(2)}")

📊 Treino: (831081, 94)
📊 Teste: (389550, 94)

🎯 Distribuição do Target (Treino):
  Treino: FPD
0    76.82
1    23.18
Name: proportion, dtype: float64


## Preparação dos Dados

In [5]:
# Backup dos dados originais
train_01 = train.copy()
test_01 = test.copy()

# lista de vars para retirar dos tratamentos
ignore_cols = ['SAFRA']

# Aplicando no treino
train_01 = train_01.drop(columns=ignore_cols)
test_01 = test_01.drop(columns=ignore_cols)

In [6]:
# Separar features e target
X_train = train_01.drop(TARGET, axis=1)
y_train = train_01[TARGET]

X_test = test_01.drop(TARGET, axis=1)
y_test = test_01[TARGET]

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')

# Garantir mesmas features em treino e teste
features_common = X_train.columns.intersection(X_test.columns)
X_train = X_train[features_common]
X_test = X_test[features_common]

print(f'\n✅ Features alinhadas: {len(features_common)}')

X_train: (831081, 92)
X_test: (389550, 92)

✅ Features alinhadas: 92


## Normalização das Features

In [7]:
'''
# Dados não normalizados
X_train_scaled = X_train
X_test_scaled = X_test
'''

'\n# Dados não normalizados\nX_train_scaled = X_train\nX_test_scaled = X_test\n'

In [8]:
# Normalizar as features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converter para DataFrame para manter nomes das features
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print('✅ Features Normalizadas')

✅ Features Normalizadas


## Validação das Premissas da Regressão Logística

### 1. Multicolinearidade (VIF)

In [9]:

# Calcular VIF para cada feature
iv_df = iv_table(train, TARGET)
iv_df.head(100)

,Variável,IV,Preditividade
1,SCORE_02,0.550328,Preditor Forte
2,SCORE_MIN,0.477611,Preditor Forte
0,SCORE_01,0.360756,Preditor Forte
30,var_62,0.101848,Preditor Moderado
16,var_30,0.093005,Preditor Fraco
...,...,...,...
7,IDADE,0.007073,Inútil para a predição
35,var_70,0.006787,Inútil para a predição
92,SAFRA,0.001741,Inútil para a predição
87,VAR25_FUNC_PUBL,0.001259,Inútil para a predição


In [10]:
# treina usando apenas as features selecionadas
selected_features = iv_df[iv_df.IV > IV].Variável.tolist()
X_train_scaled = X_train_scaled[selected_features]

KeyError: "['SAFRA'] not in index"

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'selected_features_IV_st.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(selected_features, f)

### 2. Linearidade no Logit (Log-Odds)

In [ ]:
# Testar linearidade entre features e log-odds via bins
print('\n📊 ANÁLISE DE LINEARIDADE NO LOGIT:')
print('Teste: Correlação entre feature e log(odds)\n')

linearity_results = []

for col in X_train_scaled.columns:
    # Criar bins por quantis
    bins = pd.qcut(X_train_scaled[col], q=5, duplicates='drop')

    # Agregar target por bin
    grouped = y_train.groupby(bins).agg(['sum', 'count'])
    grouped['p'] = grouped['sum'] / grouped['count']

    # Evitar log(0) e log(1)
    grouped['p'] = grouped['p'].clip(0.001, 0.999)
    grouped['log_odds'] = np.log(grouped['p'] / (1 - grouped['p']))

    # Calcular valor médio da feature por bin
    bin_midpoints = X_train_scaled[col].groupby(bins).mean()

    # Usar apenas bins válidos (onde há evento)
    valid_bins = grouped['sum'] > 0

    # Correlação feature vs log-odds (ambos no nível do bin)
    if valid_bins.sum() > 1:
        corr = np.corrcoef(
            bin_midpoints.loc[valid_bins],
            grouped.loc[valid_bins, 'log_odds']
        )[0, 1]
    else:
        corr = np.nan

    linearity_results.append({
        'Feature': col,
        'Correlação': corr
    })

linearity_df = pd.DataFrame(linearity_results)
linearity_df = linearity_df.sort_values('Correlação', key=lambda x: x.abs(), ascending=False)

print(linearity_df.to_string(index=False))
print('\n✅ Linearidade avaliada via correlação com log-odds')


In [ ]:
a

## Treinamento do Modelo - Statsmodel

In [ ]:
# aplicar pesos para balancear a classe minoritária no Logit

# Adicionar constante
X_train_sm = sm.add_constant(X_train_scaled)
X_test_sm = sm.add_constant(X_test_scaled)

# Calcular pesos por classe
n_total = len(y_train)
n_pos = y_train.sum()
n_neg = n_total - n_pos

peso_pos = n_total / (2 * n_pos)
peso_neg = n_total / (2 * n_neg)

weights = np.where(y_train == 1, peso_pos, peso_neg)

# Treinar modelo Logistic Regression com pesos
print('🤖 Treinando Regressão Logística com pesos (Statsmodels)...')
model_sm = sm.Logit(y_train, X_train_sm, freq_weights=weights).fit(disp=0)

print('\n' + '='*80)
print(model_sm.summary())
print('='*80)


## Testes Estatísticos do Modelo

In [ ]:
print('\n📊 TESTES ESTATÍSTICOS DO MODELO:')
print(f'\nLog-Likelihood: {model_sm.llf:.4f}')
print(f'AIC: {model_sm.aic:.4f}')
print(f'BIC: {model_sm.bic:.4f}')
print(f'Pseudo R-squared: {model_sm.prsquared:.4f}')

# Teste de Significância Individual (z-score)
print('\n📈 Significância Individual (p-value < 0.05):')
significant_features = model_sm.pvalues[model_sm.pvalues < 0.05]
print(f'{len(significant_features)} features significantes')

## Predições no Conjunto de Teste

In [ ]:
# Fazer predições
print('🔮 Fazendo predições no conjunto de teste...')
y_pred_proba = model_sm.predict(X_test_sm)
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f'✅ Predições concluídas!')
print(f'Proporção de classe 1: {y_pred.mean():.4f}')

## Métricas de Avaliação

In [ ]:
# Calcular métricas
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print('\n📋 RELATÓRIO DE CLASSIFICAÇÃO:')
print(classification_report(y_test, y_pred, target_names=['Bom (0)', 'Mau (1)']))

## Matriz de Confusão

In [ ]:
# Matriz de confusão
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bom (0)', 'Mau (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format='d')
plt.title('Confusion Matrix – Regressão Logística (Statsmodel)')
plt.tight_layout()
plt.show()

## Salvamento das Métricas

In [ ]:
# Criar registro de métricas
metrics_row = {
    'Precision': precision,
    'Recall': recall,
    'F1_score': f1,
    'AUC': auc,
    'TP': int(tp),
    'TN': int(tn),
    'FP': int(fp),
    'FN': int(fn),
    'Versao_Modelo': VERSAO,
    'Data_Execucao': DATA_EXECUCAO,
}

df_metrics = pd.DataFrame([metrics_row])

print('\n📊 MÉTRICAS DO MODELO:')
print(df_metrics.to_string(index=False))

## Atualizar Arquivo de Métricas

In [ ]:
# Salvar/atualizar metricas
path_metrics = METRICS_DIR / 'model_metrics.csv'

if os.path.exists(path_metrics):
    df_hist = pd.read_csv(path_metrics)
    
    # Se versão já existe, atualiza
    if VERSAO in df_hist['Versao_Modelo'].values:
        df_hist.loc[df_hist['Versao_Modelo'] == VERSAO] = df_metrics.iloc[0].values
    else:
        df_hist = pd.concat([df_hist, df_metrics], ignore_index=True)
else:
    df_hist = df_metrics

df_hist.to_csv(path_metrics, index=False)
print(f'✅ Métricas salvas em: {path_metrics}')

# Mostrar histórico de versões
print('\n📜 HISTÓRICO DE VERSÕES:')
df_hist = pd.read_csv(path_metrics)
df_hist = df_hist.sort_values('Data_Execucao', ascending=False)
print(df_hist[['Versao_Modelo', 'Precision', 'Recall', 'F1_score', 'AUC', 'Data_Execucao']].to_string(index=False))